# Regressão logística: o caso da inadimplência

Capítulo II.3 do livro vivo [Ciência de Dados e Aprendizado de Máquina](https://machinelearning.ghdaru.com.br/ii-3-regressao-logistica.html).

Leia o capítulo antes de rodar este notebook. Ele explica o porquê; aqui você vê o como, no mesmo exemplo do livro: prever se um cliente vai pagar.

Você vai:

1. tentar prever uma probabilidade com regressão linear comum, e ver onde isso quebra;
2. treinar a regressão logística com entropia cruzada;
3. ler os coeficientes como razão de chances, não como probabilidade;
4. ver o mesmo coeficiente deslocar a probabilidade de jeitos diferentes conforme o ponto de partida do cliente.

Roda sem instalar nada: só a biblioteca padrão do Python e o código do próprio livro. Nem NumPy, que só entra na etapa 09.

No fim, responda o questionário no Forms que o professor vai compartilhar na aula.

In [ ]:
# --- roda igual na sua máquina e no Colab ------------------------------
# Na sua máquina: o notebook acha o repositório subindo de pasta.
# No Colab: não há repositório, então os arquivos necessários são baixados.
import pathlib, sys, urllib.request

RAW = "https://raw.githubusercontent.com/GHDaru/machinelearning/main/"
PRECISA = ['ml-zero/etapa-05/linear.py', 'ml-zero/etapa-00/dados.py']

raiz = pathlib.Path.cwd()
for _ in range(5):
    if (raiz / "ml-zero").is_dir():
        break
    raiz = raiz.parent
else:
    raiz = pathlib.Path.cwd()

for rel in PRECISA:
    destino = raiz / rel
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(RAW + rel, destino)
        print("baixado:", rel)

sys.path.insert(0, str(raiz / "ml-zero/etapa-05"))
sys.path.insert(0, str(raiz / "ml-zero/etapa-00"))
print("pronto.")

## 1. O problema

Um banco quer prever, no momento da fatura, se um cliente vai pagar (classe 0) ou ficar inadimplente (classe 1). Três atributos: `renda_mensal`, `meses_de_relacionamento`, `uso_do_limite`.

O conjunto é sintético, com seed fixa, na mesma lógica da etapa 00. Como nós escrevemos o processo gerador, sabemos a resposta certa e podemos comparar o que o modelo encontra com o que de fato está lá. Com dado real isso não existe.

In [ ]:
from dados import gerar, dividir

dados = gerar(n=4000)
treino, validacao, teste = dividir(dados)

print(f"{len(dados)} clientes · prevalência de inadimplência {dados.prevalencia():.3f}")
print(f"treino {len(treino)} · validação {len(validacao)} · teste {len(teste)}")
print("\nprimeiro cliente:", dict(zip(dados.nomes, dados.X[0])), "-> rótulo", dados.y[0])

## 2. A tentação: usar regressão linear comum

Rotular pagou/não pagou com 0/1 e ajustar uma reta parece razoável. É só mais uma coluna numérica, não é? É o erro que o capítulo descreve antes de a sigmoide aparecer.

In [ ]:
from linear import RegressaoLinear

Xt = [[c[0], c[1], c[2]] for c in treino.X]
yt = [float(v) for v in treino.y]

linha = RegressaoLinear(solucao_fechada=True, padronizar=True).fit(Xt, yt)
pred_linha = linha.predict(Xt)

fora_do_intervalo = sum(1 for p in pred_linha if p < 0 or p > 1)
print(f"no treino: {fora_do_intervalo} de {len(pred_linha)} previsões já saem de [0,1] "
      f"({fora_do_intervalo / len(pred_linha):.1%})")

# um cliente só um pouco mais extremo que o conjunto, nada absurdo, já quebra:
ultra_seguro = [50_000.0, 240.0, 0.0]   # renda alta, 20 anos de relacionamento, limite zerado
print(f"cliente muito seguro {ultra_seguro} -> "
      f"'probabilidade' prevista de {linha.predict([ultra_seguro])[0]:+.3f}")

A reta não sabe que está prevendo uma probabilidade. Um cliente nada absurdo (renda alta, vinte anos de relacionamento, limite zerado) sai com previsão negativa, e probabilidade negativa não é probabilidade de nada. A função certa precisa saturar nas pontas. A sigmoide faz isso; a reta não.

## 3. A regressão logística de verdade

`RegressaoLogistica` (a mesma classe que o capítulo descreve) treina por gradiente, minimizando entropia cruzada. Aqui não existe solução fechada, então o gradiente não é uma opção entre outras: é o único caminho.

In [ ]:
from linear import RegressaoLogistica

Xv = [[c[0], c[1], c[2]] for c in validacao.X]
yv = [float(v) for v in validacao.y]

modelo = RegressaoLogistica(taxa=0.5, epocas=400, validacao=(Xv, yv),
                             paciencia=20, seed=0).fit(Xt, yt)

print(f"treinou {modelo.historico.epocas} épocas"
      + (" (parou cedo)" if modelo.historico.parou_cedo else ""))
print(f"perda final de treino: {modelo.historico.perdas[-1]:.4f}")

In [ ]:
Xte = [[c[0], c[1], c[2]] for c in teste.X]
yte = teste.y

pred = modelo.predict(Xte)   # limiar padrão: 0,5
acertos = sum(1 for p, y in zip(pred, yte) if p == y)
acuracia = acertos / len(yte)
baseline = 1 - teste.prevalencia()  # sempre prever a classe majoritária (0)

vp = sum(1 for p, y in zip(pred, yte) if p == 1 and y == 1)
fn = sum(1 for p, y in zip(pred, yte) if p == 0 and y == 1)
revocacao = vp / (vp + fn) if (vp + fn) else float("nan")

print(f"acurácia (limiar 0,5):   {acuracia:.4f}   (linha de base: {baseline:.4f})")
print(f"revocação da classe 1:   {revocacao:.4f}   (de {vp + fn} inadimplentes, achou {vp})")

A mesma armadilha da etapa 00. No limiar padrão a acurácia empata com a linha de base e a revocação é zero: o modelo nunca chama ninguém de inadimplente, porque quase nenhum cliente passa de 0,5 de probabilidade quando a classe positiva é só 19% do total. Não é que o modelo não aprendeu; os coeficientes da próxima seção mostram que aprendeu. É que 0,5 é o limiar errado para este problema. Baixe o limiar para a prevalência da classe:

In [ ]:
limiar = teste.prevalencia()
pred2 = [1 if p > limiar else 0 for p in modelo.predict_proba(Xte)]
acuracia2 = sum(1 for p, y in zip(pred2, yte) if p == y) / len(yte)
vp2 = sum(1 for p, y in zip(pred2, yte) if p == 1 and y == 1)
fn2 = sum(1 for p, y in zip(pred2, yte) if p == 0 and y == 1)
revocacao2 = vp2 / (vp2 + fn2)

print(f"limiar = prevalência ({limiar:.3f})")
print(f"acurácia:   {acuracia2:.4f}   (caiu)")
print(f"revocação:  {revocacao2:.4f}   (subiu bastante)")

Acurácia caiu, revocação subiu. Qual troca vale depende do custo de deixar passar um inadimplente contra o custo de incomodar um bom pagador, assunto do capítulo II.1. O que importa aqui é menor: olhando só a acurácia, você teria concluído que o modelo não aprendeu nada.

## 4. Ler o coeficiente: chance, não probabilidade

Os pesos saem na escala padronizada, "por um desvio-padrão do atributo". `razao_de_chances()` os converte: `exp(w)` é por quanto a chance de inadimplência é multiplicada.

In [ ]:
for nome, w, odds in zip(dados.nomes, modelo.pesos, modelo.razao_de_chances()):
    sinal = "aumenta" if odds > 1 else "reduz"
    print(f"{nome:24s} w={w:+.3f}   razão de chances={odds:.3f}  "
          f"({sinal} a chance em {abs(odds - 1):.0%} por desvio-padrão)")

Compare com o processo que gerou o dado (`etapa-00/dados.py`, variável `z`): `uso_do_limite` empurra para cima, `renda_mensal` (via log) e `meses_de_relacionamento` empurram para baixo. Os sinais batem, e era isso que o modelo tinha de recuperar. As magnitudes não precisam ser iguais: o processo verdadeiro usa `log(renda)`, e o modelo só viu a renda bruta padronizada.

## 5. O mesmo coeficiente, efeitos diferentes na probabilidade

É a armadilha do exercício e9 do capítulo: o coeficiente é constante na escala da chance, não na da probabilidade. Dois clientes com o mesmo aumento no uso do limite sentem efeitos diferentes se um já estava perto de 0 e o outro perto de 0,5.

In [ ]:
def cliente(renda, meses, uso):
    return [renda, meses, uso]

# dois clientes de risco bem diferente, mesmo aumento no uso do limite
baixo_risco = cliente(renda=7000, meses=60, uso=0.10)
alto_risco  = cliente(renda=2500, meses=4,  uso=0.60)

for nome, c in [("baixo risco", baixo_risco), ("alto risco", alto_risco)]:
    antes = modelo.predict_proba([c])[0]
    depois_X = [c[0], c[1], min(1.0, c[2] + 0.30)]   # +30 pontos de uso do limite
    depois = modelo.predict_proba([depois_X])[0]
    print(f"{nome:12s}  p(inadimplência) {antes:.3f} -> {depois:.3f}   "
          f"(deslocamento de {depois - antes:+.3f})")

O mesmo aumento no atributo move a probabilidade bem mais no cliente de alto risco do que no de baixo risco. É a sigmoide: íngreme no meio, achatada nas pontas. Troque os números dos dois clientes acima e veja o padrão se repetir.

## O que levar

- Regressão linear comum extrapola para fora de [0,1], porque não sabe que está prevendo uma probabilidade.
- A regressão logística não tem solução fechada. O gradiente é o único caminho.
- O coeficiente é constante na escala da chance (log-odds). Na probabilidade, o efeito depende de onde o cliente estava.
- Como o dado é sintético, dá para conferir que os sinais dos coeficientes batem com o processo gerador. Com dado real não há com o que comparar.

Volte ao capítulo [II.3 — Regressão Logística](https://machinelearning.ghdaru.com.br/ii-3-regressao-logistica.html) e responda o questionário que o professor vai compartilhar.